QR integrated with ocr & face_recognition

In [30]:
import cv2
import pytesseract
import face_recognition
import re
import os
import csv
import numpy as np
from PIL import Image, ImageTk
import tkinter as tk
from tkinter import filedialog
from tkinter import ttk
import math
import time
import qrcode
import smtplib
import socket
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email.mime.base import MIMEBase
from email import encoders
from datetime import datetime, timedelta
import random

# Configure tesseract path
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

# Email configuration
SENDER_EMAIL = "paradisebookstore888@gmail.com"
SENDER_PASSWORD = "iczp fdcl yzyr xtfs"  # Use App Password if Gmail

# Track failed attempts & lock status
failed_attempts = {}  # { email: {"count": int, "locked_until": datetime } }

# Global current image
current_image = None
verification_codes = {}  # Store verification codes temporarily

# ----------------- OCR & ID CLEANUP -----------------
def clean_student_id(ocr_id):
    # First, ensure it's uppercase
    cleaned = ocr_id.upper()
   
    # Common OCR errors to fix
    replacements = [
        ("O", "0"), ("I", "1"), ("S", "5"), ("Z", "2"),
        ("B", "8"), ("G", "6"), ("Q", "0"), (" ", ""),
        ("-", ""), (".", ""), (",", "")
    ]
   
    for wrong, right in replacements:
        cleaned = cleaned.replace(wrong, right)
   
    # Ensure proper format (e.g., 22WMR12345)
    # Extract numbers and letters separately
    numbers = re.findall(r'\d+', cleaned)
    letters = re.findall(r'[A-Z]+', cleaned)
   
    if numbers and letters:
        # Reconstruct in likely format: numbers + letters + numbers
        if len(numbers) >= 2 and len(letters) >= 1:
            return numbers[0][:2] + letters[0] + numbers[-1]
   
    return cleaned


def extract_name_and_id(image_pil):
    # Convert PIL image to OpenCV format
    img_cv = cv2.cvtColor(np.array(image_pil), cv2.COLOR_RGB2BGR)
   
    # Try multiple preprocessing techniques and rotations
    best_name = ""
    best_id = ""
    best_confidence = 0
   
    # Try different rotations (0°, 90°, 180°, 270°)
    for angle in [0, 90, 180, 270]:
        # Rotate image
        if angle == 0:
            rotated = img_cv.copy()
        elif angle == 90:
            rotated = cv2.rotate(img_cv, cv2.ROTATE_90_CLOCKWISE)
        elif angle == 180:
            rotated = cv2.rotate(img_cv, cv2.ROTATE_180)
        elif angle == 270:
            rotated = cv2.rotate(img_cv, cv2.ROTATE_90_COUNTERCLOCKWISE)
       
        # Convert to grayscale
        gray = cv2.cvtColor(rotated, cv2.COLOR_BGR2GRAY)
       
        # Try multiple preprocessing techniques
        preprocessing_methods = [
            gray, # Original grayscale
            cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1], # Otsu's thresholding
            cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2),
            cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 11, 2),
        ]
       
        for preprocessed_img in preprocessing_methods:
            # Try different OCR configurations
            ocr_configs = [
                '--psm 6', # Assume a single uniform block of text
                '--psm 4', # Assume a single column of text of variable sizes
                '--psm 8', # Treat the image as a single word
                '--psm 11', # Sparse text. Find as much text as possible in no particular order
                '--psm 12', # Sparse text with OSD
            ]
           
            for config in ocr_configs:
                # Perform OCR
                text = pytesseract.image_to_string(preprocessed_img, config=config)
                lines = [line.strip() for line in text.split("\n") if line.strip()]
               
                name = ''
                student_id = ''
               
                # Search for student ID pattern
                for i, line in enumerate(lines):
                    # Clean common OCR errors
                    line_cleaned = line.upper().replace("O", "0").replace("S", "5").replace("I", "1").replace("Z", "2")
                   
                    # Look for student ID pattern (e.g., 22WMR12345)
                    id_match = re.search(r'\d{2}[A-Z0-9]{3,5}\d{4,5}', line_cleaned)
                    if id_match:
                        raw_id = id_match.group()
                        student_id = clean_student_id(raw_id)
                       
                        # Look for name in nearby lines
                        for j in range(max(0, i-2), min(len(lines), i+3)):
                            if j == i:
                                continue # Skip the line with the ID
                               
                            candidate = lines[j].strip().upper()
                            # Filter out lines that are likely not names
                            if (len(candidate) > 3 and
                                not re.search(r'\d', candidate) and
                                len(candidate.split()) >= 2 and
                                not any(kw in candidate for kw in [
                                    'DATE', 'EXPIRY', 'STUDENT', 'TARUMT', 'MANAGEMENT',
                                    'AND', 'TECHNOLOGY', 'TUNKU', 'ABDUL', 'RAHMAN',
                                    'UNIVERSITY', 'OF', 'COLLEGE', 'ID', 'CARD'
                                ])):
                                name = candidate
                                break
                       
                        break # Found ID, stop searching
               
                # Calculate confidence (presence of ID gives high confidence)
                confidence = 2.0 if student_id else 0.5 if name else 0.0
               
                # Update best result if this is better
                if confidence > best_confidence:
                    best_confidence = confidence
                    best_name = name
                    best_id = student_id
   
    # If we found an ID but no name, try to find name in a second pass
    if best_id and not best_name:
        # Try a different approach to find the name
        for angle in [0, 90, 180, 270]:
            if angle == 0:
                rotated = img_cv.copy()
            elif angle == 90:
                rotated = cv2.rotate(img_cv, cv2.ROTATE_90_CLOCKWISE)
            elif angle == 180:
                rotated = cv2.rotate(img_cv, cv2.ROTATE_180)
            elif angle == 270:
                rotated = cv2.rotate(img_cv, cv2.ROTATE_90_COUNTERCLOCKWISE)
           
            gray = cv2.cvtColor(rotated, cv2.COLOR_BGR2GRAY)
            text = pytesseract.image_to_string(gray, config='--psm 6')
            lines = [line.strip() for line in text.split("\n") if line.strip()]
           
            for line in lines:
                candidate = line.strip().upper()
                if (len(candidate) > 3 and
                    not re.search(r'\d', candidate) and
                    len(candidate.split()) >= 2 and
                    not any(kw in candidate for kw in [
                        'DATE', 'EXPIRY', 'STUDENT', 'TARUMT', 'MANAGEMENT',
                        'AND', 'TECHNOLOGY', 'TUNKU', 'ABDUL', 'RAHMAN',
                        'UNIVERSITY', 'OF', 'COLLEGE', 'ID', 'CARD'
                    ])):
                    best_name = candidate
                    break
           
            if best_name:
                break
     # DEBUG: Print extracted text if no good result found
    if not best_id:
        print("DEBUG: Could not extract ID. Extracted text from all rotations:")
        for angle in [0, 90, 180, 270]:
            if angle == 0:
                rotated = img_cv.copy()
            elif angle == 90:
                rotated = cv2.rotate(img_cv, cv2.ROTATE_90_CLOCKWISE)
            elif angle == 180:
                rotated = cv2.rotate(img_cv, cv2.ROTATE_180)
            elif angle == 270:
                rotated = cv2.rotate(img_cv, cv2.ROTATE_90_COUNTERCLOCKWISE)
           
            gray = cv2.cvtColor(rotated, cv2.COLOR_BGR2GRAY)
            text = pytesseract.image_to_string(gray, config='--psm 6')
            print(f"Angle {angle}°: {text}")
           
    return best_name, best_id
   
# ----------------- QR CODE GENERATION -----------------
def generate_qr_code(student_name, student_id, email, folder_name):
    qr_data = f"Name: {student_name}\nID: {student_id}\nEmail: {email}"
    qr = qrcode.QRCode(version=1, box_size=10, border=5)
    qr.add_data(qr_data)
    qr.make(fit=True)
    img_qr = qr.make_image(fill_color="black", back_color="white")
    qr_path = os.path.join(folder_name, f"{student_name}.{student_id}_qr.png")
    img_qr.save(qr_path)
    return qr_path

# ----------------- EMAIL VALIDATION -----------------
def is_valid_email(email):
    if not re.match(r"[^@]+@[^@]+\.[^@]+", email):
        return False
    domain = email.split("@")[1]
    try:
        socket.gethostbyname(domain)
        return True
    except socket.error:
        return False

# ----------------- EMAIL SENDING -----------------
def send_email_with_qr(to_email, student_name, qr_path, status_label):
    try:
        subject = "🎓 Your Graduation QR Code"
        body = f"Dear {student_name},\n\nPlease find attached your unique QR code for the graduation ceremony.\nBring this QR code with you for scanning during the event.\n\nRegards,\nGraduation Committee"
        msg = MIMEMultipart()
        msg["From"] = SENDER_EMAIL
        msg["To"] = to_email
        msg["Subject"] = subject
        msg.attach(MIMEText(body, "plain"))
        with open(qr_path, "rb") as f:
            mime = MIMEBase("image", "png", filename=os.path.basename(qr_path))
            mime.add_header("Content-Disposition", "attachment", filename=os.path.basename(qr_path))
            mime.add_header("X-Attachment-Id", "0")
            mime.add_header("Content-ID", "<0>")
            mime.set_payload(f.read())
            encoders.encode_base64(mime)
            msg.attach(mime)
        server = smtplib.SMTP("smtp.gmail.com", 587)
        server.starttls()
        server.login(SENDER_EMAIL, SENDER_PASSWORD)
        server.sendmail(SENDER_EMAIL, to_email, msg.as_string())
        server.quit()
        status_label.config(text=f"QR Code sent to {to_email} successfully!", foreground="green")
        return True
    except Exception as e:
        status_label.config(text=f"Failed to send email: {e}", foreground="red")
        return False

# ----------------- EMAIL VERIFICATION -----------------
def send_verification_code(to_email, status_label):
    if to_email in failed_attempts:
        info = failed_attempts[to_email]
        if info["count"] >= 3 and datetime.now() < info["locked_until"]:
            status_label.config(text=f"Too many failed attempts for {to_email}. Try again later.", foreground="red")
            return False
    try:
        code = str(random.randint(100000, 999999))
        verification_codes[to_email] = code
        subject = "🎓 Your Email Verification Code"
        body = f"Dear Student,\n\nYour verification code for convocation registration is: {code}\n\nDo not share this code with anyone."
        msg = MIMEMultipart()
        msg["From"] = SENDER_EMAIL
        msg["To"] = to_email
        msg["Subject"] = subject
        msg.attach(MIMEText(body, "plain"))
        server = smtplib.SMTP("smtp.gmail.com", 587)
        server.starttls()
        server.login(SENDER_EMAIL, SENDER_PASSWORD)
        server.sendmail(SENDER_EMAIL, to_email, msg.as_string())
        server.quit()
        status_label.config(text=f"A verification code has been sent to {to_email}.", foreground="green")
        return True
    except Exception as e:
        status_label.config(text=f"Failed to send verification code: {e}", foreground="red")
        return False

# ----------------- GUI FUNCTIONS -----------------
def display_image_pil(pil_img, image_panel):
    global current_image
    current_image = pil_img
    img_resized = pil_img.resize((350, 250))
    img_tk = ImageTk.PhotoImage(img_resized)
    image_panel.config(image=img_tk)
    image_panel.image = img_tk

def upload_image(image_panel, status_label, upload_btn, capture_btn, confirm_btn, retake_btn):
    file_path = filedialog.askopenfilename(filetypes=[("Image files", "*.jpg *.jpeg *.png")])
    if file_path:
        pil_img = Image.open(file_path)
        display_image_pil(pil_img, image_panel)
        status_label.config(text="Image uploaded. Confirm or retake.", foreground="blue")
        confirm_btn.pack(pady=10)
        retake_btn.pack(pady=5)
        upload_btn.pack_forget()
        capture_btn.pack_forget()

def take_picture(image_panel, status_label, upload_btn, capture_btn, confirm_btn, retake_btn):
    cap = cv2.VideoCapture(0)
    frame_width = int(cap.get(3))
    frame_height = int(cap.get(4))
    rect_w, rect_h = 480, 300
    rect_x = (frame_width - rect_w) // 2
    rect_y = (frame_height - rect_h) // 2
    captured_frame = None
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        cv2.rectangle(frame, (rect_x, rect_y), (rect_x + rect_w, rect_y + rect_h), (0, 255, 0), 2)
        cv2.putText(frame, "Press SPACE to capture, ESC to quit", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 2)
        cv2.imshow("Capture ID Card", frame)
        key = cv2.waitKey(1) & 0xFF
        if key == 32:
            captured_frame = frame[rect_y:rect_y + rect_h, rect_x:rect_x + rect_w]
            break
        elif key == 27:
            break
    cap.release()
    cv2.destroyAllWindows()
    if captured_frame is not None:
        pil_img = Image.fromarray(cv2.cvtColor(captured_frame, cv2.COLOR_BGR2RGB))
        display_image_pil(pil_img, image_panel)
        status_label.config(text="Image captured. Confirm or retake.", foreground="blue")
        confirm_btn.pack(pady=10)
        retake_btn.pack(pady=5)
        upload_btn.pack_forget()
        capture_btn.pack_forget()

def confirm_image(image_panel, status_label, confirm_btn, retake_btn, edit_frame, verify_frame):
    global current_image
    if current_image is None:
        status_label.config(text="Please select or capture an image first.", foreground="red")
        return
    
    name, student_id = extract_name_and_id(current_image)
   
    if not name and not student_id:
        status_label.config(text="Could not extract name or student ID. Please try again.", foreground="red")
        return
   
    # Clear the image from the panel
    image_panel.config(image=None)
    image_panel.image = None
   
    status_label.config(text="Extracted info. Please confirm or edit below.", foreground="green")
    edit_frame.pack(pady=10)
    confirm_btn.pack_forget()
    retake_btn.pack_forget()
   
    def force_uppercase_name(*args):
        current = name_var.get()
        name_var.set(current.upper())
   
    name_var = tk.StringVar()
    name_var.trace_add("write", force_uppercase_name)
   
    def force_uppercase_id(*args):
        current = id_var.get()
        id_var.set(current.upper())
   
    id_var = tk.StringVar()
    id_var.trace_add("write", force_uppercase_id)
   
    def force_lowercase_email(*args):
        current = email_var.get()
        email_var.set(current.lower())
   
    email_var = tk.StringVar()
    email_var.trace_add("write", force_lowercase_email)
   
    ttk.Label(edit_frame, text="Name:").grid(row=0, column=0, padx=10, pady=5, sticky="e")
    name_entry = ttk.Entry(edit_frame, width=40, textvariable=name_var)
    name_var.set(name)
    name_entry.grid(row=0, column=1, padx=10, pady=5)
   
    ttk.Label(edit_frame, text="Student ID:").grid(row=1, column=0, padx=10, pady=5, sticky="e")
    id_entry = ttk.Entry(edit_frame, width=40, textvariable=id_var)
    corrected_id = clean_student_id(student_id)
    id_var.set(corrected_id)
    id_entry.grid(row=1, column=1, padx=10, pady=5)
   
    ttk.Label(edit_frame, text="School Email:").grid(row=2, column=0, padx=10, pady=5, sticky="e")
    email_entry = ttk.Entry(edit_frame, width=40, textvariable=email_var)
    email_entry.grid(row=2, column=1, padx=10, pady=5)
    email_entry.focus_set()
   
    def save_data():
        final_name = name_entry.get().strip().replace(" ", "_")
        final_id = id_entry.get().strip()
        email = email_entry.get().strip()
        if not final_name or not final_id or not email:
            status_label.config(text="Name, Student ID, and Email are required.", foreground="red")
            return
       
        id_pattern = r"^\d{2}[A-Z]{3}\d{5}$"
        if not re.match(id_pattern, final_id):
            status_label.config(text="Invalid Student ID format. Example: 24WMR00000", foreground="red")
            id_entry.focus_set()
            return
        
        school_email_pattern = r"^[a-z0-9]+-[a-z]{2}[0-9]{2}@student\.tarc\.edu\.my$"
        if not re.match(school_email_pattern, email.lower()):
            status_label.config(text="Please enter a valid school email.", foreground="red")
            email_entry.focus_set()
            return
        file_exists = os.path.exists("student_records.csv")
        if file_exists:
            with open("student_records.csv", "r", newline="") as fr:
                existing_emails = [row["Email"] for row in csv.DictReader(fr)]
            if email in existing_emails:
                status_label.config(text="This email has already been verified.", foreground="red")
                email_entry.focus_set()
                return
        if not send_verification_code(email, status_label):
            retake_or_reselect(image_panel, status_label, upload_btn, capture_btn, confirm_btn, retake_btn, edit_frame, verify_frame)
            return
       
        edit_frame.pack_forget()
        verify_frame.pack(pady=10)
       
        ttk.Label(verify_frame, text=f"Enter code sent to {email}:").grid(row=0, column=0, columnspan=2, pady=10)
        code_var = tk.StringVar()
        code_entry = ttk.Entry(verify_frame, textvariable=code_var)
        code_entry.grid(row=1, column=0, columnspan=2, padx=10, pady=5)
       
        def verify_code():
            user_code = code_var.get().strip()
            correct_code = verification_codes.get(email)
            if user_code == correct_code:
                failed_attempts[email] = {"count": 0, "locked_until": datetime.min}
                status_label.config(text="Email verified successfully! Proceeding to face capture.", foreground="green")
                verify_frame.pack_forget()
                proceed_with_registration(final_name, final_id, email, status_label, image_panel, upload_btn, capture_btn, confirm_btn, retake_btn, edit_frame, verify_frame)
            else:
                if email not in failed_attempts:
                    failed_attempts[email] = {"count": 0, "locked_until": datetime.min}
                failed_attempts[email]["count"] += 1
                remaining = 3 - failed_attempts[email]["count"]
                if remaining > 0:
                    status_label.config(text=f"Wrong code. {remaining} attempt(s) left.", foreground="red")
                else:
                    failed_attempts[email]["locked_until"] = datetime.now() + timedelta(minutes=5)
                    status_label.config(text="Too many failed attempts. Locked for 5 minutes.", foreground="red")
                    verify_frame.pack_forget()
                    edit_frame.pack_forget()
                    retake_or_reselect(image_panel, status_label, upload_btn, capture_btn, confirm_btn, retake_btn, edit_frame, verify_frame)
       
        ttk.Button(verify_frame, text="Verify", command=verify_code).grid(row=2, column=0, columnspan=2, pady=10)
   
    ttk.Button(edit_frame, text="Confirm & Save", command=save_data).grid(row=3, column=0, columnspan=2, pady=10)

def proceed_with_registration(final_name, final_id, email, status_label, image_panel, upload_btn, capture_btn, confirm_btn, retake_btn, edit_frame, verify_frame):
    folder_name = os.path.join("StudentidFolder", f"{final_name}.{final_id}")
    os.makedirs(folder_name, exist_ok=True)
    img_path = os.path.join(folder_name, f"{final_name}.{final_id}.jpg")
    current_image.save(img_path)
    
    temp_folder_created = False
    frame_count = 0
    buffer_encodings = []
    capture_interval = 10
    timeout_seconds = 60
    start_time = time.time()
    cap = cv2.VideoCapture(0)
    from mtcnn import MTCNN
    detector = MTCNN()
    while True:
        ret, frame = cap.read()
        if not ret:
            continue
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = detector.detect_faces(rgb)
        h, w, _ = frame.shape
        center = (w // 2, h // 2)
        radius = 140
        segments = 40
        completed_ratio = len(buffer_encodings) / 7
        for i in range(segments):
            angle = 2 * math.pi * i / segments
            x1 = int(center[0] + radius * math.cos(angle))
            y1 = int(center[1] + radius * math.sin(angle))
            x2 = int(center[0] + (radius + 12) * math.cos(angle))
            y2 = int(center[1] + (radius + 12) * math.sin(angle))
            if i < math.floor(completed_ratio * segments):
                cv2.line(frame, (x1, y1), (x2, y2), (0, 200, 0), 2)
            else:
                cv2.line(frame, (x1, y1), (x2, y2), (180, 180, 180), 2)
        cv2.circle(frame, center, radius - 10, (255, 255, 255), 2)
        cv2.putText(frame, "Move your head slowly to complete the circle", (center[0] - 200, center[1] + radius + 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        if len(results) == 1:
            if not temp_folder_created:
                os.makedirs(folder_name, exist_ok=True)
                temp_folder_created = True
                img_path = os.path.join(folder_name, f"{final_name}.{final_id}.jpg")
                current_image.save(img_path)
            if frame_count % capture_interval == 0:
                x, y, w_box, h_box = results[0]['box']
                top, right, bottom, left = y, x + w_box, y + h_box, x
                encodings = face_recognition.face_encodings(rgb, known_face_locations=[(top, right, bottom, left)])
                if encodings:
                    buffer_encodings.append(encodings[0])
                    if len(buffer_encodings) >= 7:
                        mean_encoding = np.mean(buffer_encodings, axis=0)
                        np.save(os.path.join(folder_name, "face_encoding.npy"), mean_encoding)
                        file_exists = os.path.exists("student_records.csv")

                        with open("student_records.csv", "a", newline="") as f:
                            writer = csv.writer(f)
                            if not file_exists or os.stat("student_records.csv").st_size == 0:
                                writer.writerow(["Name", "Student ID", "Email", "Image Path"])
                            writer.writerow([final_name.replace("_", " "), final_id, email, img_path])

                        status_label.config(text="Face capture complete! Generating QR code.", foreground="green")
                        break
        elif len(results) > 1:
            cv2.putText(frame, "Multiple faces detected", (60, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
        frame_count += 1
        if time.time() - start_time > timeout_seconds:
            if temp_folder_created:
                try:
                    for f in os.listdir(folder_name):
                        os.remove(os.path.join(folder_name, f))
                    os.rmdir(folder_name)
                except:
                    pass
            status_label.config(text="Timeout: No face detected. Please try again.", foreground="red")
            break
        cv2.imshow("Live Face Capture", frame)
        if cv2.waitKey(1) & 0xFF == 27:
            break
    cap.release()
    cv2.destroyAllWindows()
    qr_path = generate_qr_code(final_name, final_id, email, folder_name)
    send_email_with_qr(email, final_name, qr_path, status_label)
    retake_or_reselect(image_panel, status_label, upload_btn, capture_btn, confirm_btn, retake_btn, edit_frame, verify_frame)

def retake_or_reselect(image_panel, status_label, upload_btn, capture_btn, confirm_btn, retake_btn, edit_frame, verify_frame):
    global current_image
    image_panel.config(image=None)
    image_panel.image = None
    current_image = None
    edit_frame.pack_forget()
    verify_frame.pack_forget()
    confirm_btn.pack_forget()
    retake_btn.pack_forget()
    upload_btn.pack(pady=10)
    capture_btn.pack(pady=5)
    status_label.config(text="Ready for new registration.", foreground="blue")

# ----------------- GUI SETUP -----------------
root = tk.Tk()
root.title("🎓 Convocation Registration System")
root.geometry("480x700")
root.configure(bg="#f0f8ff")  # Light blue background

style = ttk.Style()
style.theme_use('clam')
style.configure("TButton", background="#add8e6", foreground="black", font=("Helvetica", 10))
style.configure("TLabel", background="#f0f8ff", foreground="#00008b", font=("Helvetica", 10))
style.map("TButton", background=[('active', '#87ceeb')])
style.configure("TEntry", fieldbackground="#f0f8ff", background="#f0f8ff")
style.configure("Custom.TFrame", background="#f0f8ff")

welcome = ttk.Label(root, text="Welcome to Convocation Registration System!", font=("Helvetica", 14, "bold"))
welcome.pack(pady=10)

instruction = ttk.Label(root, text="Please register using your student ID card (upload or webcam)")
instruction.pack()

status_label = ttk.Label(root, text="Status: Ready", foreground="blue", font=("Helvetica", 10, "italic"))
status_label.pack(pady=5)

upload_btn = ttk.Button(root, text="Upload Student ID Card")
upload_btn.pack(pady=10)

capture_btn = ttk.Button(root, text="Capture Student ID via Webcam")
capture_btn.pack(pady=5)

image_panel = ttk.Label(root, background="#f0f8ff")
image_panel.pack(padx=10, pady=10)

confirm_btn = ttk.Button(root, text="Confirm ID Card")
retake_btn = ttk.Button(root, text="Retake / Select Another Image")

edit_frame = ttk.Frame(root, padding=10, style="Custom.TFrame")  # Apply custom style
edit_frame.configure(relief="raised")

verify_frame = ttk.Frame(root, padding=10, style="Custom.TFrame")
verify_frame.configure(relief="raised")

# Bind commands after defining frames
upload_btn.config(command=lambda: upload_image(image_panel, status_label, upload_btn, capture_btn, confirm_btn, retake_btn))
capture_btn.config(command=lambda: take_picture(image_panel, status_label, upload_btn, capture_btn, confirm_btn, retake_btn))
confirm_btn.config(command=lambda: confirm_image(image_panel, status_label, confirm_btn, retake_btn, edit_frame, verify_frame))
retake_btn.config(command=lambda: retake_or_reselect(image_panel, status_label, upload_btn, capture_btn, confirm_btn, retake_btn, edit_frame, verify_frame))

root.mainloop()

Admin Assign CGPA to Student

In [31]:
import tkinter as tk
from tkinter import ttk, messagebox
import csv
import os
import re

# ----------------- Admin UI for CGPA and Award Management -----------------
def load_student_records():
    """Load student records from CSV into a list of dictionaries."""
    student_records = []
    if os.path.exists("student_records.csv"):
        try:
            with open("student_records.csv", "r", newline="") as f:
                reader = csv.DictReader(f)
                for row in reader:
                    student_records.append({
                        "Name": row.get("Name", ""),
                        "Student ID": row.get("Student ID", ""),
                        "Email": row.get("Email", ""),
                        "Image Path": row.get("Image Path", ""),
                        "CGPA": row.get("CGPA", ""),
                        "Award_Category": row.get("Award_Category", ""),
                        "Category_Sequence": row.get("Category_Sequence", "")
                    })
        except Exception as e:
            messagebox.showerror("Error", f"Failed to load student_records.csv: {e}")
    return student_records

def assign_award_category(cgpa):
    """Assign award category based on CGPA."""
    try:
        cgpa = float(cgpa)
        if cgpa >= 3.75:
            return "DISTINCTION"
        elif cgpa >= 2.75:
            return "MERIT"
        else:
            return ""
    except ValueError:
        return ""

def update_category_sequences(students):
    """Assign sequence numbers within each award category."""
    category_groups = {"DISTINCTION": [], "MERIT": []}
    for student in students:
        category = student["Award_Category"]
        if category in category_groups:
            category_groups[category].append(student)

    for category in category_groups:
        for index, student in enumerate(sorted(category_groups[category], key=lambda x: float(x["CGPA"] or 0), reverse=True)):
            student["Category_Sequence"] = str(index + 1)

def save_student_records(students, status_label):
    """Save updated student records to CSV."""
    fieldnames = ["Name", "Student ID", "Email", "Image Path", "CGPA", "Award_Category", "Category_Sequence"]
    try:
        with open("student_records.csv", "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            for student in students:
                writer.writerow(student)
        status_label.config(text="Successfully saved to student_records.csv", foreground="green")
    except Exception as e:
        status_label.config(text=f"Failed to save to student_records.csv: {e}", foreground="red")
        messagebox.showerror("Error", f"Failed to save to student_records.csv: {e}")
        raise

def create_admin_ui():
    root = tk.Tk()
    root.title("🎓 Admin CGPA and Award Manager")
    root.geometry("800x600")
    root.configure(bg="#f0f8ff")  # Light blue background to match Convocation Registration System

    # Configure ttk style to match the original UI
    style = ttk.Style()
    style.theme_use('clam')
    style.configure("TButton", background="#add8e6", foreground="black", font=("Helvetica", 10))
    style.configure("TLabel", background="#f0f8ff", foreground="#00008b", font=("Helvetica", 10))
    style.map("TButton", background=[('active', '#87ceeb')])
    style.configure("Treeview", font=("Helvetica", 10))
    style.configure("Treeview.Heading", font=("Helvetica", 10, "bold"))

    # Welcome and instructions
    ttk.Label(root, text="Admin: Manage CGPA and Award Categories", font=("Helvetica", 14, "bold")).pack(pady=10)
    ttk.Label(root, text="Enter CGPA (exactly 4 decimal places, e.g., 3.1234) for each student to save.").pack(pady=5)

    # Status label
    status_label = ttk.Label(root, text="Status: Ready", foreground="blue", font=("Helvetica", 10, "italic"))
    status_label.pack(pady=5)

    # Load student records
    students = load_student_records()
    if not students:
        messagebox.showwarning("No Records", "No student records found in student_records.csv.")
        root.destroy()
        return

    # Treeview for displaying and editing student data
    columns = ("Name", "Student ID", "Email", "CGPA", "Award_Category", "Category_Sequence")
    tree_frame = ttk.Frame(root)
    tree_frame.pack(padx=10, pady=10, fill="both", expand=True)
    tree = ttk.Treeview(tree_frame, columns=columns, show="headings", height=15)
    tree.pack(side="left", fill="both", expand=True)

    # Add scrollbar
    scrollbar = ttk.Scrollbar(tree_frame, orient="vertical", command=tree.yview)
    scrollbar.pack(side="right", fill="y")
    tree.configure(yscrollcommand=scrollbar.set)

    # Set column headings
    tree.heading("Name", text="Name")
    tree.heading("Student ID", text="Student ID")
    tree.heading("Email", text="Email")
    tree.heading("CGPA", text="CGPA")
    tree.heading("Award_Category", text="Award Category")
    tree.heading("Category_Sequence", text="Category Sequence")

    # Set column widths
    tree.column("Name", width=150)
    tree.column("Student ID", width=100)
    tree.column("Email", width=200)
    tree.column("CGPA", width=80)
    tree.column("Award_Category", width=120)
    tree.column("Category_Sequence", width=120)

    # Populate treeview with student data
    for student in students:
        tree.insert("", "end", values=(
            student["Name"],
            student["Student ID"],
            student["Email"],
            student["CGPA"],
            student["Award_Category"],
            student["Category_Sequence"]
        ))

    # CGPA entry and update button
    def update_cgpa():
        selected_item = tree.selection()
        if not selected_item:
            status_label.config(text="Please select a student to update CGPA.", foreground="red")
            messagebox.showwarning("No Selection", "Please select a student to update CGPA.")
            return

        cgpa_window = tk.Toplevel(root)
        cgpa_window.title("Update CGPA")
        cgpa_window.geometry("300x150")
        cgpa_window.configure(bg="#f0f8ff")
        cgpa_window.transient(root)
        cgpa_window.grab_set()

        ttk.Label(cgpa_window, text="Enter CGPA (0.0000 - 4.0000):").pack(pady=10)
        cgpa_var = tk.StringVar()
        cgpa_entry = ttk.Entry(cgpa_window, textvariable=cgpa_var)
        cgpa_entry.pack(pady=5)
        cgpa_entry.focus_set()

        def save_cgpa():
            try:
                cgpa = cgpa_var.get().strip()
                if not re.match(r"^\d\.\d{4}$", cgpa):
                    status_label.config(text="CGPA must have exactly 4 decimal places (e.g., 3.1234).", foreground="red")
                    messagebox.showerror("Invalid CGPA", "CGPA must have exactly 4 decimal places (e.g., 3.1234).", parent=cgpa_window)
                    return
                cgpa_float = float(cgpa)
                if not 0.0000 <= cgpa_float <= 4.0000:
                    status_label.config(text="CGPA must be between 0.0000 and 4.0000.", foreground="red")
                    messagebox.showerror("Invalid CGPA", "CGPA must be between 0.0000 and 4.0000.", parent=cgpa_window)
                    return
                student_name = tree.item(selected_item)["values"][0]
                selected_id = tree.item(selected_item)["values"][1]
                for student in students:
                    if student["Student ID"] == selected_id:
                        student["CGPA"] = cgpa
                        student["Award_Category"] = assign_award_category(cgpa)
                        break
                update_category_sequences(students)
                save_student_records(students, status_label)
                # Update treeview
                for item in tree.get_children():
                    tree.delete(item)
                for student in students:
                    tree.insert("", "end", values=(
                        student["Name"],
                        student["Student ID"],
                        student["Email"],
                        student["CGPA"],
                        student["Award_Category"],
                        student["Category_Sequence"]
                    ))
                status_label.config(text=f"CGPA updated for {student_name} and saved to CSV.", foreground="green")
                messagebox.showinfo("Success", f"CGPA updated for {student_name} and saved to CSV.")
                cgpa_window.grab_release()
                cgpa_window.destroy()
            except Exception as e:
                status_label.config(text=f"Failed to save CGPA: {e}", foreground="red")
                messagebox.showerror("Error", f"Failed to save CGPA: {e}", parent=cgpa_window)
                cgpa_window.grab_release()
                cgpa_window.destroy()

        ttk.Button(cgpa_window, text="Save CGPA", command=save_cgpa).pack(pady=10)

    # Button frame for consistent layout
    button_frame = ttk.Frame(root)
    button_frame.pack(pady=5)
    ttk.Button(button_frame, text="Update Selected Student's CGPA", command=update_cgpa).pack(side="left", padx=5)
    ttk.Button(button_frame, text="Exit", command=root.destroy).pack(side="left", padx=5)

    root.mainloop()

if __name__ == "__main__":
    create_admin_ui()

DURING CEREMONY QR + FACE

In [32]:
import cv2
import numpy as np
import face_recognition
import csv
from pyzbar import pyzbar
import tkinter as tk
from tkinter import messagebox, ttk
import time
import os
import pyttsx3

# ----------------- Helper for popups -----------------
def show_popup(title, message, parent, status_label):
    """Display a popup with consistent styling and update status label."""
    popup = tk.Toplevel(parent)
    popup.title(f"🎓 {title}")
    popup.geometry("250x80")
    popup.configure(bg="#f0f8ff")
    ttk.Label(popup, text=message, wraplength=200, font=("Helvetica", 10)).pack(pady=5)
    ttk.Button(popup, text="OK", command=popup.destroy).pack(pady=5)
    popup.transient(parent)
    popup.grab_set()
    status_label.config(text=message, foreground="blue" if "Verified" in title else "red")
    parent.wait_window(popup)

# ----------------- Select Award Category -----------------
def select_award_category(parent, status_label):
    """Prompt user to select an award category with consistent UI styling."""
    category_window = tk.Toplevel(parent)
    category_window.title("🎓 Select Award Category")
    category_window.geometry("300x150")
    category_window.configure(bg="#f0f8ff")
    
    ttk.Label(category_window, text="Select Award Category for Ceremony:", font=("Helvetica", 12)).pack(pady=10)
    
    categories = ["DISTINCTION", "MERIT"]
    category_var = tk.StringVar(value=categories[0])  # Default to first category
    category_dropdown = ttk.Combobox(category_window, textvariable=category_var, values=categories, state="readonly", font=("Helvetica", 10))
    category_dropdown.pack(pady=10)
    
    def confirm_selection():
        selected = category_var.get()
        if selected:
            parent.selected_category = selected
            parent.category_selected = True
            status_label.config(text=f"Selected category: {selected}", foreground="green")
            category_window.destroy()
            category_window.quit()
        else:
            status_label.config(text="Please select a category.", foreground="red")
            messagebox.showerror("Error", "Please select a category.", parent=category_window)
    
    ttk.Button(category_window, text="Confirm", command=confirm_selection).pack(pady=10)
    category_window.transient(parent)
    category_window.grab_set()
    category_window.mainloop()
    return category_var.get()

# ----------------- Load students for selected category -----------------
def load_students_for_category(selected_category, student_encodings, students_list, attendance_marked_students):
    student_encodings.clear()
    students_list.clear()
    attendance_marked_students.clear()
    
    student_records_file = "student_records.csv"
    if not os.path.exists(student_records_file):
        print(f"Error: {student_records_file} does not exist.")
        return False
    
    student_found = False
    with open(student_records_file, "r") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row.get('Award_Category') != selected_category:
                continue
            student_found = True
            name = row.get('Name', '').replace(" ", "_")
            sid = row.get('Student ID', '')
            image_path = row.get('Image Path', '')
            sequence = row.get('Category_Sequence', '')
            folder_key = f"{name}.{sid}"
            encoding_path = os.path.join("StudentidFolder", folder_key, "face_encoding.npy")
           
            print(f"Processing student: {folder_key}, Award_Category: {row.get('Award_Category')}")
           
            if os.path.exists(encoding_path):
                try:
                    encoding = np.load(encoding_path)
                    student_encodings[folder_key] = encoding
                    students_list.append({'name': name, 'sid': sid, 'key': folder_key, 'sequence': sequence})
                    print(f"Loaded face encoding for {folder_key}")
                except Exception as e:
                    print(f"Failed to load face encoding for {folder_key}: {e}")
            else:
                print(f"No face encoding file found for {folder_key} at {encoding_path}")
           
            # Check if image path exists (for debugging, not used for encoding)
            if not os.path.exists(image_path):
                print(f"Image not found for {folder_key} at {image_path}")
   
    # Load marked students for this category from attendance.csv
    attendance_file = "attendance.csv"
    if os.path.exists(attendance_file):
        with open(attendance_file, "r") as f:
            next(f) # skip header
            for line in f:
                cols = line.strip().split(",")
                if len(cols) >= 3 and cols[2] == "Present":
                    folder_key = f"{cols[0].replace(' ', '_')}.{cols[1]}"
                    if folder_key in student_encodings:
                        attendance_marked_students.add(folder_key)
                        print(f"Marked as present: {folder_key}")
   
    print(f"Loaded {len(students_list)} students for category {selected_category}")
    return student_found

# ----------------- TTS --------------------------
def talk_student_name(student_name, sequence, category):
    try:
        engine = pyttsx3.init()
        engine.setProperty('rate', 150)
        engine.setProperty('volume', 0.8)
        announcement = f"{student_name}, {category}"
        print(f"Speaking: {announcement}")
        engine.say(announcement)
        engine.runAndWait()
    except Exception as e:
        print(f"TTS error: {e}")
    finally:
        if 'engine' in locals():
            engine.stop()

# ----------------- Main Program -----------------
def main():
    student_records_file = "student_records.csv"
    attendance_file = "attendance.csv"
    student_encodings = {}
    students_list = []
    attendance_marked_students = set()

    # Initialize Tkinter root
    root = tk.Tk()
    root.title("🎓 Student Check-in System")
    root.geometry("400x300")
    root.configure(bg="#f0f8ff")  # Light blue background to match Convocation Registration System
    
    # Configure ttk style
    style = ttk.Style()
    style.theme_use('clam')
    style.configure("TButton", background="#add8e6", foreground="black", font=("Helvetica", 10))
    style.configure("TLabel", background="#f0f8ff", foreground="#00008b", font=("Helvetica", 10))
    style.map("TButton", background=[('active', '#87ceeb')])

    # Welcome and instructions
    ttk.Label(root, text="Student Check-in System", font=("Helvetica", 14, "bold")).pack(pady=10)
    ttk.Label(root, text="Scan QR code and verify face for attendance.", font=("Helvetica", 10)).pack(pady=5)
    
    # Status label
    status_label = ttk.Label(root, text="Status: Ready", foreground="blue", font=("Helvetica", 10, "italic"))
    status_label.pack(pady=5)

    # Select initial category with retry mechanism
    while True:
        select_award_category(root, status_label)
        if not root.category_selected:
            status_label.config(text="No category selected. Exiting.", foreground="red")
            print("No category selected. Exiting.")
            root.destroy()
            return
        selected_category = root.selected_category
        category_label = ttk.Label(root, text=f"Current Category: {selected_category}", font=("Helvetica", 12))
        category_label.pack(pady=10)
        
        # Load students for selected category
        students_found = load_students_for_category(selected_category, student_encodings, students_list, attendance_marked_students)
        
        if not students_found:
            show_popup("No Students Found", f"No registered students found for {selected_category}. Please select another category.", root, status_label)
            print(f"=== No registered students found for {selected_category}. ===")
            continue  # Prompt for category selection again
        break  # Exit loop if students are found
    
    # Setup attendance file
    if not os.path.exists(attendance_file):
        with open(attendance_file, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["Name", "Student ID", "Status"])
            for student in students_list:
                writer.writerow([student['name'].replace("_", " "), student['sid'], "Absent"])
    else:
        print("📄 Using existing attendance.csv (will keep previous marks).")
    
    # Check if students were loaded (redundant check, kept for safety)
    if not student_encodings:
        show_popup("Error", f"No registered students found for {selected_category}. Exiting.", root, status_label)
        print(f"=== No registered student encodings found for {selected_category}. ===")
        root.destroy()
        return

    # Webcam setup
    cap = cv2.VideoCapture(0)
    font = cv2.FONT_HERSHEY_SIMPLEX

    qr_verified_student = None
    current_step = "QR Scan"
    face_match_start_time = None
    countdown_time = 3  # seconds needed to confirm face
    status_label.config(text=f"Scan QR code for {selected_category} category.", foreground="blue")
    print(f"📷 Webcam started. Please scan your QR code inside the box for {selected_category} category.")

    def end_session():
        nonlocal selected_category, qr_verified_student, current_step, face_match_start_time, cap
        cap.release()
        cv2.destroyAllWindows()
        root.category_selected = False
        while True:
            select_award_category(root, status_label)
            if not root.category_selected:
                status_label.config(text="No category selected. Exiting.", foreground="red")
                print("No category selected. Exiting.")
                root.destroy()
                root.quit()
                return
            selected_category = root.selected_category
            category_label.config(text=f"Current Category: {selected_category}")
            students_found = load_students_for_category(selected_category, student_encodings, students_list, attendance_marked_students)
            if not students_found:
                show_popup("No Students Found", f"No registered students found for {selected_category}. Please select another category.", root, status_label)
                print(f"=== No registered students found for {selected_category}. ===")
                continue
            break
        if not student_encodings:
            show_popup("Error", f"No registered students found for {selected_category}. Exiting.", root, status_label)
            print(f"=== No registered student encodings found for {selected_category}. ===")
            root.destroy()
            root.quit()
            return
        qr_verified_student = None
        current_step = "QR Scan"
        face_match_start_time = None
        status_label.config(text=f"Scan QR code for {selected_category} category.", foreground="blue")
        print(f"📷 Webcam restarted for {selected_category} category.")
        cap = cv2.VideoCapture(0)

    while True:
        ret, frame = cap.read()
        if not ret:
            status_label.config(text="Webcam error. Please check connection.", foreground="red")
            continue

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        h, w, _ = frame.shape
        y_offset = 30

        # --- Define boxes ---
        qr_box_h = h // 3
        qr_box_w = w // 3
        qr_box_top = (h - qr_box_h) // 2
        qr_box_bottom = qr_box_top + qr_box_h
        qr_box_left = (w - qr_box_w) // 2
        qr_box_right = qr_box_left + qr_box_w

        face_box_h = h * 2 // 3
        face_box_w = w * 2 // 3
        face_box_top = (h - face_box_h) // 2
        face_box_bottom = face_box_top + face_box_h
        face_box_left = (w - face_box_w) // 2
        face_box_right = face_box_left + face_box_w

        # Draw boxes
        if current_step == "QR Scan":
            cv2.rectangle(frame, (qr_box_left, qr_box_top), (qr_box_right, qr_box_bottom), (0, 255, 255), 2)
        else:
            cv2.rectangle(frame, (face_box_left, face_box_top), (face_box_right, face_box_bottom), (255, 255, 0), 2)

        # ===== QR SCANNING PHASE =====
        if current_step == "QR Scan" and qr_verified_student is None:
            cv2.putText(frame, f"Align QR inside the box ({selected_category})", (qr_box_left, qr_box_top - 10),
                        font, 0.6, (0, 255, 255), 2)

            decoded_qrs = pyzbar.decode(frame)
            for qr in decoded_qrs:
                (x, y, qr_w, qr_h) = qr.rect
                qr_center_x = x + qr_w // 2
                qr_center_y = y + qr_h // 2

                if qr_box_left < qr_center_x < qr_box_right and qr_box_top < qr_center_y < qr_box_bottom:
                    qr_data = qr.data.decode("utf-8")
                    lines = qr_data.split("\n")
                    try:
                        name_line = [l for l in lines if "Name:" in l][0]
                        id_line = [l for l in lines if "ID:" in l][0]
                        student_name = name_line.split(":")[1].strip().replace(" ", "_")
                        student_id = id_line.split(":")[1].strip()
                        folder_key = f"{student_name}.{student_id}"

                        if folder_key in student_encodings:
                            if folder_key in attendance_marked_students:
                                status_label.config(text=f"Attendance already marked for {student_name.replace('_',' ')} ({student_id})", foreground="red")
                                root.update()  # Force GUI update
                            else:
                                qr_verified_student = folder_key
                                current_step = "Face Scan"
                                status_label.config(text=f"QR Verified! {student_name.replace('_',' ')} ({student_id})", foreground="green")
                                root.update()  # Force GUI update
                        else:
                            status_label.config(text=f"This student is not in the {selected_category} category!", foreground="red")
                            root.update()  # Force GUI update
                    except:
                        status_label.config(text="QR Code format is not valid!", foreground="red")
                        root.update()  # Force GUI update

        # ===== FACE RECOGNITION PHASE =====
        elif current_step == "Face Scan" and qr_verified_student:
            face_locations = face_recognition.face_locations(rgb)
            face_encodings = face_recognition.face_encodings(rgb, face_locations)

            matched = False
            for (top, right, bottom, left), live_encoding in zip(face_locations, face_encodings):
                face_center_x = (left + right) // 2
                face_center_y = (top + bottom) // 2
                if not (face_box_left < face_center_x < face_box_right and face_box_top < face_center_y < face_box_bottom):
                    cv2.putText(frame, "Keep face inside the box!", (10, h - 20), font, 0.7, (0, 0, 255), 2)
                    continue

                db_encoding = student_encodings[qr_verified_student]
                distance = face_recognition.face_distance([db_encoding], live_encoding)[0]

                if distance < 0.6:
                    cv2.rectangle(frame, (left, top), (right, bottom), (0, 255, 0), 3)
                    cv2.putText(frame, "Face Match", (left, top - 10), font, 0.7, (0, 255, 0), 2)
                    matched = True
                else:
                    cv2.rectangle(frame, (left, top), (right, bottom), (0, 0, 255), 3)
                    cv2.putText(frame, "Not matching!", (left, top - 10), font, 0.6, (0, 0, 255), 2)

            # Handle countdown if matched
            if matched:
                if face_match_start_time is None:
                    face_match_start_time = time.time()
                elapsed = int(time.time() - face_match_start_time)
                remaining = countdown_time - elapsed

                if remaining > 0:
                    cv2.putText(frame, f"Hold still... {remaining}", (50, h - 50), font, 0.8, (0, 255, 0), 2)
                else:
                    # Update attendance to Present
                    name, sid = qr_verified_student.split(".", 1)
                    sequence = next(s['sequence'] for s in students_list if s['key'] == qr_verified_student)
                    rows = []
                    with open(attendance_file, "r") as f:
                        reader = csv.reader(f)
                        rows = list(reader)
                    with open(attendance_file, "w", newline="") as f:
                        writer = csv.writer(f)
                        for row in rows:
                            if len(row) >= 3 and row[0].replace(" ", "_") == name and row[1] == sid:
                                row[2] = "Present"
                            writer.writerow(row)

                    attendance_marked_students.add(qr_verified_student)
                    status_label.config(text=f"Attendance marked for {name.replace('_',' ')} ({sid})", foreground="green")
                    root.update()  # Force GUI update
                    talk_student_name(name.replace('_', ' '), sequence, selected_category)

                    qr_verified_student = None
                    current_step = "QR Scan"
                    face_match_start_time = None
                    status_label.config(text=f"Scan QR code for {selected_category} category.", foreground="blue")
                root.update()  # Force GUI update
            else:
                face_match_start_time = None

        # ===== DISPLAY STATUS =====
        status_text = f"Step: {current_step} ({selected_category})"
        cv2.putText(frame, status_text, (10, y_offset), font, 0.7, (0, 0, 0), 2)

        if qr_verified_student:
            name, sid = qr_verified_student.split(".", 1)
            sequence = next(s['sequence'] for s in students_list if s['key'] == qr_verified_student)
            cv2.putText(frame, f"{name.replace('_',' ')} - {sid}",
                        (10, y_offset + 30), font, 0.7, (0, 255, 0), 2)

        cv2.putText(frame, "Press Q to end session, ESC to quit", (10, y_offset + 70), font, 0.6, (0, 0, 0), 2)
        cv2.imshow("Student Check-in System", frame)

        # Handle keypresses
        key = cv2.waitKey(1) & 0xFF
        if key == 27:  # ESC to quit
            break
        elif key == ord('q') or key == ord('Q'):  # Q to end session
            end_session()

    cap.release()
    cv2.destroyAllWindows()
    root.destroy()

if __name__ == "__main__":
    main()

Processing student: WONG_WAI_FENG.24WMR09307, Award_Category: MERIT
Loaded face encoding for WONG_WAI_FENG.24WMR09307
Loaded 1 students for category MERIT
📷 Webcam started. Please scan your QR code inside the box for MERIT category.
Speaking: WONG WAI FENG, MERIT
